In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, learning_curve
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import joblib



train = pd.read_csv(r"Dataset/PM_train.csv")
test = pd.read_csv(r"Dataset/PM_test.csv")
truth = pd.read_csv(r"Dataset/PM_truth.csv")
print(train.head())
print("Train shape: ",train.shape)



# Creating RUL {Remaining Useful Life}
max_cycle = train.groupby('id')['cycle'].max().reset_index()
max_cycle.columns = ['id','max_cycle']

train = train.merge(max_cycle, on='id')

train['RUL'] = train['max_cycle'] - train['cycle']

train.drop('max_cycle', axis=1, inplace=True)
print("\nDataset with RUL:")
print(train.head())

# Cap RUL
RUL_CAP = 125
train['RUL'] = train['RUL'].clip(upper=RUL_CAP)



# Data cleaning (removing flat sensor readings)
sensor_cols = [col for col in train.columns if col.startswith('s')]

sensor_std = train[sensor_cols].std()
print("\nSensor Standard Deviations")
print(sensor_std)

flat_sensors = sensor_std[sensor_std < 0.01].index.tolist()
print("Flat Sensors Removed:", flat_sensors)

train.drop(columns=flat_sensors, inplace=True)
test.drop(columns=flat_sensors, inplace=True)



# Temporal features 
sensor_cols = [col for col in train.columns if col.startswith('s')]

for col in sensor_cols:
    train[col+"_mean"] = train.groupby('id')[col].rolling(5).mean().reset_index(0,drop=True)
    train[col+"_std"] = train.groupby('id')[col].rolling(5).std().reset_index(0,drop=True)
train.fillna(method='bfill', inplace=True)



# Additional Feature Engineering
max_cycle = train.groupby('id')['cycle'].transform('max')
train['cycle_ratio'] = train['cycle'] / max_cycle

# Sensor degradation trend
for col in sensor_cols:
    train[col+"_diff"] = train.groupby('id')[col].diff()
train.fillna(method="bfill", inplace=True)

# Define features
X = train.drop(columns=['id','cycle','RUL'])
y = train['RUL']
groups = train['id']



# Group based Train-test split
gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)

for train_idx, val_idx in gss.split(X, y, groups):

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)



# Random Forest
rf = RandomForestRegressor(
    n_estimators = 300,
    max_depth = 20,
    random_state = 42
)
rf.fit(X_train_scaled, y_train)

rf_train_pred = rf.predict(X_train_scaled)
rf_val_pred = rf.predict(X_val_scaled)

print("\nRandom Forest Training Performance")
print("MAE:",mean_absolute_error(y_train,rf_train_pred))
print("RMSE:",np.sqrt(np.mean((y_train-rf_train_pred)**2)))
print("R2:",r2_score(y_train,rf_train_pred))

print("\nRandom Forest Validation Performance")
print("MAE:",mean_absolute_error(y_val,rf_val_pred))
print("RMSE:",np.sqrt(np.mean((y_val-rf_val_pred)**2)))
print("R2:",r2_score(y_val,rf_val_pred))



# XGBoost model
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.2,
    reg_lambda=1,
    random_state=42
)

xgb_model.fit(X_train_scaled, y_train)

xgb_train_pred = xgb_model.predict(X_train_scaled)
xgb_val_pred = xgb_model.predict(X_val_scaled)

print("\nXGBoost Training Performance")
print("MAE:",mean_absolute_error(y_train,xgb_train_pred))
print("RMSE:",np.sqrt(np.mean((y_train-xgb_train_pred)**2)))
print("R2:",r2_score(y_train,xgb_train_pred))

print("\nXGBoost Validation Performance")
print("MAE:",mean_absolute_error(y_val,xgb_val_pred))
print("RMSE:",np.sqrt(np.mean((y_val-xgb_val_pred)**2)))
print("R2:",r2_score(y_val,xgb_val_pred))



# Actual vs Predicted Visualization
plt.figure(figsize=(6,6))
plt.scatter(y_val,xgb_val_pred,alpha=0.5)
plt.xlabel("Actual RUL")
plt.ylabel("Predicted RUL")
plt.title("Actual vs Predicted RUL")
plt.plot([0,125],[0,125])
plt.show()



# Feature Importance
importance = xgb_model.feature_importances_

feature_importance = pd.DataFrame({
    'Feature':X.columns,
    'Importance':importance

}).sort_values(by='Importance',ascending=False)

print("\nTop Important Features")
print(feature_importance.head(15))



plt.figure(figsize=(10,5))

feature_importance.head(15).plot(
    x="Feature",
    y="Importance",
    kind="bar"
)

plt.title("Top 15 Important Features")
plt.xticks(rotation=45)
plt.show()



joblib.dump(xgb_model, "model_rul.pkl")
joblib.dump(scaler, "scaler.pkl")
print("Model and scaler saved successfully")